In [6]:
import pandas as pd
import os 
from sqlalchemy import create_engine
import logging
import time

logging.basicConfig(
        filename = "logs/ingestion_db.log",
        level = logging.DEBUG,
        format ="%(asctime)s -%(levelname)s - %(message)s",
        filemode="a"
)
    
engine = create_engine('sqlite:///inventory.db') 

def ingest_db(df, table_name, engine):
    '''this funtion will ingest the dataframe into database table'''
    df.to_sql(table_name, con = engine, if_exists = 'replace', index = False)

def load_raw_data():
    '''this function will load the CSVs as dataframe and ingest into db'''
    
    start =time.time()

    for file in os.listdir(r'C:\Users\hp\OneDrive\Desktop\Data'):
        if '.csv' in file :
            df = pd.read_csv(r'C:\Users\hp\OneDrive\Desktop\Data/'+file)
            logging.info(f'Ingesting{file} in db')
            ingest_db(df,file[:-4], engine)
            
    end = time.time()
    total_time = (end - start)/60
    
    logging.info('Ingestion complete')
    logging.info(f'Total Time Taken: {total_time} minutes')
    
if __name__ == '__main__':

    
    load_raw_data()



# Exploratory Data Analysis

Understanding the dataset to explore how the data is present in the present in the database and if there is need of creating some aggregated tables that can help with:

- Vendor selection  for probitability

- Product Pricing Optimization

In [7]:
import pandas as pd
import sqlite3


In [8]:
# creating database connnection
conn=sqlite3.connect('inventory.db')
            

In [9]:
# checking tables present inthe database

tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type ='table'", conn)

tables

,name
0,begin_inventory
1,end_inventory
2,purchases
3,purchase_prices
4,sales
5,vendor_invoice


In [5]:
for table in tables ['name'] :
    print ('-'* 50, f'{table}','-'*50)
    print( 'counts of records:', pd.read_sql(f" select count(*) as count from {table}",conn) ['count'].values[0]) 

-------------------------------------------------- begin_inventory --------------------------------------------------
counts of records: 206529
-------------------------------------------------- end_inventory --------------------------------------------------
counts of records: 224489
-------------------------------------------------- purchases --------------------------------------------------
counts of records: 1048575
-------------------------------------------------- purchase_prices --------------------------------------------------
counts of records: 12261
-------------------------------------------------- sales --------------------------------------------------
counts of records: 1048575
-------------------------------------------------- vendor_invoice --------------------------------------------------
counts of records: 5543


In [16]:
for table in tables ['name'] :
    print ('-'* 50, f'{table}','-'*50)
    print( 'counts of records:', pd.read_sql(f" select count(*) as count from {table}",conn) ['count'].values[0]) 
    display(pd.read_sql(f"select * from {table} limit 5", conn))

-------------------------------------------------- begin_inventory --------------------------------------------------
counts of records: 206529


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,startDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,8,12.99,01-01-2024
1,1_HARDERSFIELD_60,1,HARDERSFIELD,60,Canadian Club 1858 VAP,750mL,7,10.99,01-01-2024
2,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,6,36.99,01-01-2024
3,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,3,38.99,01-01-2024
4,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,6,34.99,01-01-2024


-------------------------------------------------- end_inventory --------------------------------------------------
counts of records: 224489


,InventoryId,Store,City,Brand,Description,Size,onHand,Price,endDate
0,1_HARDERSFIELD_58,1,HARDERSFIELD,58,Gekkeikan Black & Gold Sake,750mL,11,12.99,31-12-2024
1,1_HARDERSFIELD_62,1,HARDERSFIELD,62,Herradura Silver Tequila,750mL,7,36.99,31-12-2024
2,1_HARDERSFIELD_63,1,HARDERSFIELD,63,Herradura Reposado Tequila,750mL,7,38.99,31-12-2024
3,1_HARDERSFIELD_72,1,HARDERSFIELD,72,No. 3 London Dry Gin,750mL,4,34.99,31-12-2024
4,1_HARDERSFIELD_75,1,HARDERSFIELD,75,Three Olives Tomato Vodka,750mL,7,14.99,31-12-2024


-------------------------------------------------- purchases --------------------------------------------------
counts of records: 1048575


...

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,58,Gekkeikan Black & Gold Sake,12.99,750mL,750,1,9.28,8320,SHAW ROSS INT L IMP LTD
1,62,Herradura Silver Tequila,36.99,750mL,750,1,28.67,1128,BROWN-FORMAN CORP
2,63,Herradura Reposado Tequila,38.99,750mL,750,1,30.46,1128,BROWN-FORMAN CORP
3,72,No. 3 London Dry Gin,34.99,750mL,750,1,26.11,9165,ULTRA BEVERAGE COMPANY LLP
4,75,Three Olives Tomato Vodka,14.99,750mL,750,1,10.94,7245,PROXIMO SPIRITS INC.


-------------------------------------------------- sales --------------------------------------------------
counts of records: 1048575


,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,01-01-2024,750,1,0.79,12546,JIM BEAM BRANDS COMPANY
1,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,2,32.98,16.49,02-01-2024,750,1,1.57,12546,JIM BEAM BRANDS COMPANY
2,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,16.49,16.49,03-01-2024,750,1,0.79,12546,JIM BEAM BRANDS COMPANY
3,1_HARDERSFIELD_1004,1,1004,Jim Beam w/2 Rocks Glasses,750mL,1,14.49,14.49,08-01-2024,750,1,0.79,12546,JIM BEAM BRANDS COMPANY
4,1_HARDERSFIELD_1005,1,1005,Maker's Mark Combo Pack,375mL 2 Pk,2,69.98,34.99,09-01-2024,375,1,0.79,12546,JIM BEAM BRANDS COMPANY


-------------------------------------------------- vendor_invoice --------------------------------------------------
counts of records: 5543


,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,105,ALTAMAR BRANDS LLC,04-01-2024,8124,21-12-2023,16-02-2024,6,214.26,3.47,None
1,4466,AMERICAN VINTAGE BEVERAGE,07-01-2024,8137,22-12-2023,21-02-2024,15,140.55,8.57,None
2,388,ATLANTIC IMPORTING COMPANY,09-01-2024,8169,24-12-2023,16-02-2024,5,106.60,4.61,None
3,480,BACARDI USA INC,12-01-2024,8106,20-12-2023,05-02-2024,10100,137483.78,2935.20,None
4,516,BANFI PRODUCTS CORP,07-01-2024,8170,24-12-2023,12-02-2024,1935,15527.25,429.20,None


In [18]:
purchases = pd.read_sql_query("SELECT * from purchases WHERE VendorNumber =4466", conn)
purchases


,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,22-12-2023,01-01-2024,07-01-2024,21-02-2024,9.35,4,37.40,1
1,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,22-12-2023,02-01-2024,07-01-2024,21-02-2024,9.41,5,47.05,1
2,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,22-12-2023,01-01-2024,07-01-2024,21-02-2024,9.35,6,56.10,1
3,38_GOULCREST_5215,38,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,27-12-2023,07-01-2024,19-01-2024,26-02-2024,9.41,6,56.46,1
4,59_CLAETHORPES_5215,59,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,27-12-2023,05-01-2024,19-01-2024,26-02-2024,9.41,6,56.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
860,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,10777,22-06-2024,28-06-2024,09-07-2024,15-08-2024,9.41,4,37.64,1
861,60_IRRAGIN_5215,60,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,10777,22-06-2024,30-06-2024,09-07-2024,15-08-2024,9.41,5,47.05,1
862,60_IRRAGIN_5255,60,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,10777,22-06-2024,30-06-2024,09-07-2024,15-08-2024,9.35,2,18.70,1
863,60_IRRAGIN_3140,60,3140,TGI Fridays Orange Dream,1.75L,4466,AMERICAN VINTAGE BEVERAGE,10777,22-06-2024,30-06-2024,09-07-2024,15-08-2024,11.19,6,67.14,1


In [19]:
purchase_prices = pd.read_sql_query("SELECT * from purchase_prices WHERE VendorNumber =4466", conn)
purchase_prices

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,5215,TGI Fridays Long Island Iced,12.99,1750mL,1750,1,9.41,4466,AMERICAN VINTAGE BEVERAGE
1,5255,TGI Fridays Ultimte Mudslide,12.99,1750mL,1750,1,9.35,4466,AMERICAN VINTAGE BEVERAGE
2,3140,TGI Fridays Orange Dream,14.99,1750mL,1750,1,11.19,4466,AMERICAN VINTAGE BEVERAGE


In [20]:
vendor_invoice  = pd.read_sql_query("SELECT * from  vendor_invoice WHERE VendorNumber =4466", conn)
vendor_invoice 

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,07-01-2024,8137,22-12-2023,21-02-2024,15,140.55,8.57,None
1,4466,AMERICAN VINTAGE BEVERAGE,19-01-2024,8207,27-12-2023,26-02-2024,335,3142.33,16.97,None
2,4466,AMERICAN VINTAGE BEVERAGE,18-01-2024,8307,03-01-2024,18-02-2024,41,383.35,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,27-01-2024,8469,14-01-2024,11-03-2024,72,673.20,3.30,None
4,4466,AMERICAN VINTAGE BEVERAGE,04-02-2024,8532,19-01-2024,15-03-2024,79,740.21,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,09-02-2024,8604,24-01-2024,15-03-2024,347,3261.37,17.61,None
6,4466,AMERICAN VINTAGE BEVERAGE,17-02-2024,8793,05-02-2024,02-04-2024,72,675.36,3.17,None
7,4466,AMERICAN VINTAGE BEVERAGE,01-03-2024,8892,12-02-2024,28-03-2024,117,1096.05,5.15,None
8,4466,AMERICAN VINTAGE BEVERAGE,07-03-2024,8995,19-02-2024,02-04-2024,129,1209.27,5.44,None
9,4466,AMERICAN VINTAGE BEVERAGE,12-03-2024,9033,22-02-2024,16-04-2024,147,1377.87,6.61,None


In [36]:
sales = pd.read_sql_query("SELECT * from  sales WHERE VendorNo =4466", conn)
sales

,InventoryId,Store,Brand,Description,Size,SalesQuantity,SalesDollars,SalesPrice,SalesDate,Volume,Classification,ExciseTax,VendorNo,VendorName
0,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,09-01-2024,1750,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
1,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,12-01-2024,1750,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
2,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,15-01-2024,1750,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
3,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,21-01-2024,1750,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
4,1_HARDERSFIELD_5215,1,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,23-01-2024,1750,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
426,15_WANBORNE_5215,15,5215,TGI Fridays Long Island Iced,1.75L,1,12.99,12.99,23-02-2024,1750,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
427,15_WANBORNE_5255,15,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,03-02-2024,1750,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
428,15_WANBORNE_5255,15,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,23-02-2024,1750,1,1.84,4466,AMERICAN VINTAGE BEVERAGE
429,15_WANBORNE_5255,15,5255,TGI Fridays Ultimte Mudslide,1.75L,1,12.99,12.99,24-02-2024,1750,1,1.84,4466,AMERICAN VINTAGE BEVERAGE


In [20]:
purchases.groupby(['Brand', 'PurchasePrice'])[['Quantity', 'Dollars']].sum()

,,Quantity,Dollars
Brand,PurchasePrice,,
3140,11.19,1552,17366.88
5215,9.41,2267,21332.47
5255,9.35,2135,19962.25


In [21]:
purchase_prices

,Brand,Description,Price,Size,Volume,Classification,PurchasePrice,VendorNumber,VendorName
0,5215,TGI Fridays Long Island Iced,12.99,1750mL,1750,1,9.41,4466,AMERICAN VINTAGE BEVERAGE
1,5255,TGI Fridays Ultimte Mudslide,12.99,1750mL,1750,1,9.35,4466,AMERICAN VINTAGE BEVERAGE
2,3140,TGI Fridays Orange Dream,14.99,1750mL,1750,1,11.19,4466,AMERICAN VINTAGE BEVERAGE


In [22]:
vendor_invoice

,VendorNumber,VendorName,InvoiceDate,PONumber,PODate,PayDate,Quantity,Dollars,Freight,Approval
0,4466,AMERICAN VINTAGE BEVERAGE,07-01-2024,8137,22-12-2023,21-02-2024,15,140.55,8.57,None
1,4466,AMERICAN VINTAGE BEVERAGE,19-01-2024,8207,27-12-2023,26-02-2024,335,3142.33,16.97,None
2,4466,AMERICAN VINTAGE BEVERAGE,18-01-2024,8307,03-01-2024,18-02-2024,41,383.35,1.99,None
3,4466,AMERICAN VINTAGE BEVERAGE,27-01-2024,8469,14-01-2024,11-03-2024,72,673.20,3.30,None
4,4466,AMERICAN VINTAGE BEVERAGE,04-02-2024,8532,19-01-2024,15-03-2024,79,740.21,3.48,None
5,4466,AMERICAN VINTAGE BEVERAGE,09-02-2024,8604,24-01-2024,15-03-2024,347,3261.37,17.61,None
6,4466,AMERICAN VINTAGE BEVERAGE,17-02-2024,8793,05-02-2024,02-04-2024,72,675.36,3.17,None
7,4466,AMERICAN VINTAGE BEVERAGE,01-03-2024,8892,12-02-2024,28-03-2024,117,1096.05,5.15,None
8,4466,AMERICAN VINTAGE BEVERAGE,07-03-2024,8995,19-02-2024,02-04-2024,129,1209.27,5.44,None
9,4466,AMERICAN VINTAGE BEVERAGE,12-03-2024,9033,22-02-2024,16-04-2024,147,1377.87,6.61,None


In [24]:
vendor_invoice['PONumber'].nunique()

55

In [25]:
vendor_invoice.shape

(55, 10)

In [21]:
vendor_invoice.columns

Index(['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate',
       'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval'],
      dtype='object')

In [27]:
purchases

,InventoryId,Store,Brand,Description,Size,VendorNumber,VendorName,PONumber,PODate,ReceivingDate,InvoiceDate,PayDate,PurchasePrice,Quantity,Dollars,Classification
0,30_CULCHETH_5255,30,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,22-12-2023,01-01-2024,07-01-2024,21-02-2024,9.35,4,37.40,1
1,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,22-12-2023,02-01-2024,07-01-2024,21-02-2024,9.41,5,47.05,1
2,1_HARDERSFIELD_5255,1,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8137,22-12-2023,01-01-2024,07-01-2024,21-02-2024,9.35,6,56.10,1
3,38_GOULCREST_5215,38,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,27-12-2023,07-01-2024,19-01-2024,26-02-2024,9.41,6,56.46,1
4,59_CLAETHORPES_5215,59,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,8207,27-12-2023,05-01-2024,19-01-2024,26-02-2024,9.41,6,56.46,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
860,34_PITMERDEN_5215,34,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,10777,22-06-2024,28-06-2024,09-07-2024,15-08-2024,9.41,4,37.64,1
861,60_IRRAGIN_5215,60,5215,TGI Fridays Long Island Iced,1.75L,4466,AMERICAN VINTAGE BEVERAGE,10777,22-06-2024,30-06-2024,09-07-2024,15-08-2024,9.41,5,47.05,1
862,60_IRRAGIN_5255,60,5255,TGI Fridays Ultimte Mudslide,1.75L,4466,AMERICAN VINTAGE BEVERAGE,10777,22-06-2024,30-06-2024,09-07-2024,15-08-2024,9.35,2,18.70,1
863,60_IRRAGIN_3140,60,3140,TGI Fridays Orange Dream,1.75L,4466,AMERICAN VINTAGE BEVERAGE,10777,22-06-2024,30-06-2024,09-07-2024,15-08-2024,11.19,6,67.14,1


In [28]:
sales.groupby('Brand')[['SalesDollars','SalesPrice','SalesQuantity']].sum()

,SalesDollars,SalesPrice,SalesQuantity
Brand,,,
5215,3897.00,2416.14,300
5255,4273.71,3182.55,329


- The purchases table contains actual purchase data, including the date of purchase , products(brands) purchased by vendors, the amount paid(in dollars), and the quantity.
- The purchase price column is derived from the purchase_prices table, which provided product-wise actual and purchase prices.The combination of vendor and brand is unique in this table.
- The vendor_invoice table aggregates data from the purchases table, summarizing quantity and dollar amounts, along with an additional column for freight.
This table maintains  uniqueness based on vendor and PO number.
- The sales table captures actual sales transctions, detailing the brands purchased by vendors, the quantity sold ,the 
selling price, and the revenue earned.

---

As the data that we need for analysis is distributed in different tables, we need to create a summary table containing :

- purchase transction
- sales transaction data
- freight costs for each vendor
- actual product prices from vendors

In [22]:
vendor_invoice.columns

Index(['VendorNumber', 'VendorName', 'InvoiceDate', 'PONumber', 'PODate',
       'PayDate', 'Quantity', 'Dollars', 'Freight', 'Approval'],
      dtype='object')

In [27]:
freight_summary = pd.read_sql_query("""
SELECT VendorName, SUM(Freight) AS FreightCost
FROM vendor_invoice
GROUP BY VendorName
""", conn)

In [28]:
freight_summary

,VendorName,FreightCost
0,AAPER ALCOHOL & CHEMICAL CO,0.48
1,ADAMBA IMPORTS INTL INC,367.52
2,ALISA CARR BEVERAGES,172.00
3,ALTAMAR BRANDS LLC,62.39
4,AMERICAN SPIRITS EXCHANGE,6.19
...,...,...
124,WEIN BAUER INC,222.23
125,WESTERN SPIRITS BEVERAGE CO,1933.19
126,WILLIAM GRANT & SONS INC,30234.42
127,WINE GROUP INC,27100.41


In [29]:
purchases.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'VendorNumber',
       'VendorName', 'PONumber', 'PODate', 'ReceivingDate', 'InvoiceDate',
       'PayDate', 'PurchasePrice', 'Quantity', 'Dollars', 'Classification'],
      dtype='object')

In [30]:
purchase_prices.columns

Index(['Brand', 'Description', 'Price', 'Size', 'Volume', 'Classification',
       'PurchasePrice', 'VendorNumber', 'VendorName'],
      dtype='object')

In [33]:
pd.read_sql_query(""" Select
p.VendorNumber,
p.VendorName,
p.Brand,
p.PurchasePrice,
pp.Volume,
pp.Price as ActualPrice,
SUM(p.Quantity) as TotalPurchaseQuantity,
SUM(p.Dollars) as TotalPurchaseDollars
FROM purchases p
JOIN purchase_prices PP
ON p.Brand = pp.Brand
Where p.PurchasePrice >0
GROUP BY p.VendorNumber , p.VendorName, p.Brand
ORDER BY  TotalPurchaseDollars""", conn)

,VendorNumber,VendorName,Brand,PurchasePrice,Volume,ActualPrice,TotalPurchaseQuantity,TotalPurchaseDollars
0,7245,PROXIMO SPIRITS INC.,3065,0.71,50,0.99,1,0.71
1,3960,DIAGEO NORTH AMERICA INC,3775,0.73,50,0.99,1,0.73
2,9815,WINE GROUP INC,22407,2.25,750,3.29,1,2.25
3,8004,SAZERAC CO INC,5683,0.39,50,0.49,6,2.34
4,9815,WINE GROUP INC,8527,1.32,750,4.99,2,2.64
...,...,...,...,...,...,...,...,...
8507,3960,DIAGEO NORTH AMERICA INC,3545,21.89,1750,29.99,58783,1286759.87
8508,17035,PERNOD RICARD USA,8068,18.24,1750,24.99,75385,1375022.40
8509,4425,MARTIGNETTI COMPANIES,3405,23.19,1750,28.99,62385,1446708.15
8510,3960,DIAGEO NORTH AMERICA INC,4261,16.17,1750,22.99,96073,1553500.41


In [38]:
sales.columns

Index(['InventoryId', 'Store', 'Brand', 'Description', 'Size', 'SalesQuantity',
       'SalesDollars', 'SalesPrice', 'SalesDate', 'Volume', 'Classification',
       'ExciseTax', 'VendorNo', 'VendorName'],
      dtype='object')

In [40]:
pd.read_sql_query("""
SELECT 
    VendorNo, 
    Brand, 
    SUM(SalesDollars) AS TotalSalesDollars,
    SUM(SalesQuantity) AS TotalSalesQuantity,
    SUM(ExciseTax) AS TotalExciseTax
FROM sales
GROUP BY VendorNo, Brand
ORDER BY TotalSalesDollars
""", conn)

,VendorNo,Brand,TotalSalesDollars,TotalSalesQuantity,TotalExciseTax
0,8004,5287,0.98,2,0.10
1,3960,3303,0.99,1,0.05
2,9206,2773,0.99,1,0.05
3,9625,8872,0.99,1,0.05
4,3252,3933,1.98,2,0.10
...,...,...,...,...,...
7653,4425,3405,275162.97,9203,16909.12
7654,17035,8068,288135.11,11189,20557.97
7655,1128,1233,344712.22,9578,17598.14
7656,3960,3545,357759.17,11883,21833.58


In [44]:

import time

start = time.time()

final_table = pd.read_sql_query("""
SELECT
    pp.VendorNumber,
    pp.Brand,
    pp.Price AS ActualPrice,
    pp.PurchasePrice,
    
    SUM(s.SalesDollars) AS TotalSalesDollars,
    SUM(s.SalesQuantity) AS TotalSalesQuantity,
    SUM(s.ExciseTax) AS TotalExciseTax,
    SUM(s.SalesPrice) AS TotalSalesPrice,
    
    SUM(vi.Quantity) AS TotalPurchaseQuantity,
    SUM(vi.Dollars) AS TotalPurchaseDollars,
    SUM(vi.Freight) AS TotalFreightCost

FROM purchase_prices pp

JOIN sales s
    ON pp.VendorNumber = s.VendorNo
    AND pp.Brand = s.Brand

JOIN vendor_invoice vi
    ON pp.VendorNumber = vi.VendorNumber

GROUP BY 
    pp.VendorNumber, 
    pp.Brand, 
    pp.Price, 
    pp.PurchasePrice
""", conn)

end = time.time()

print(f"Time taken: {(end - start)/60} minutes")


Time taken: 5.615012470881144 minutes


In [19]:
vendor_sales_summary = pd.read_sql_query("""
WITH FreightSummary AS (
    SELECT 
        VendorNumber,
        SUM(Freight) AS FreightCost
    FROM vendor_invoice
    GROUP BY VendorNumber
),

PurchaseSummary AS (
    SELECT
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.Volume,
        pp.Price AS ActualPrice,
        SUM(p.Quantity) AS TotalPurchaseQuantity,
        SUM(p.Dollars) AS TotalPurchaseDollars
    FROM purchases p
    JOIN purchase_prices pp
        ON p.Brand = pp.Brand
    WHERE p.PurchasePrice > 0
    GROUP BY 
        p.VendorNumber,
        p.VendorName,
        p.Brand,
        p.Description,
        p.PurchasePrice,
        pp.Volume,
        pp.Price
),

SalesSummary AS (
    SELECT 
        VendorNo, 
        Brand, 
        SUM(SalesDollars) AS TotalSalesDollars,
        SUM(SalesQuantity) AS TotalSalesQuantity,
        SUM(ExciseTax) AS TotalExciseTax
    FROM sales
    GROUP BY VendorNo, Brand
)

SELECT
    ps.VendorNumber,
    ps.VendorName,
    ps.Brand,
    ps.Description,
    ps.PurchasePrice,
    ps.ActualPrice,
    ps.Volume,
    ps.TotalPurchaseQuantity,
    ps.TotalPurchaseDollars,
    ss.TotalSalesQuantity,
    ss.TotalSalesDollars,
    ss.TotalExciseTax,
    fs.FreightCost

FROM PurchaseSummary ps

LEFT JOIN SalesSummary ss
    ON ps.VendorNumber = ss.VendorNo
    AND ps.Brand = ss.Brand

LEFT JOIN FreightSummary fs
    ON ps.VendorNumber = fs.VendorNumber

ORDER BY ps.TotalPurchaseDollars DESC
""", conn)

In [10]:
vendor_sales_summary

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750,60320,1584606.40,9578.0,344712.22,17598.14,68601.68
1,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750,96073,1553500.41,20226.0,444810.74,37163.76,257032.07
2,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750,62385,1446708.15,9203.0,275162.97,16909.12,144929.24
3,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750,75385,1375022.40,11189.0,288135.11,20557.97,123780.22
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750,58783,1286759.87,11883.0,357759.17,21833.58,257032.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8507,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,4.99,750,2,2.64,3.0,5.97,0.33,27100.41
8508,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,0.49,50,6,2.34,128.0,62.72,6.72,50293.62
8509,9815,WINE GROUP INC,22407,Three Wishes Chard,2.25,3.29,750,1,2.25,1.0,3.29,0.11,27100.41
8510,3960,DIAGEO NORTH AMERICA INC,3775,Smirnoff Sorbet Pine/Coconut,0.73,0.99,50,1,0.73,NaN,NaN,NaN,257032.07


This query generates a vendor-wise sales and purchase summary.

### Performance Optimization

- The query involves heavy joins and aggregations on large datasets like sales and purchases.
- Storing the pre-aggregated results avoids repeated expensive computations.
- Helps in analyzing sales, purchases, and pricing for different vendors and brands.
- Future Benefits of storing this data for fastert fopr Dashboarding and Reporting.
- Instead of running expensive queries each time, dashboards can fetch data quickly from vendor_sales_summary


In [11]:
vendor_sales_summary.dtypes

VendorNumber               int64
VendorName                object
Brand                      int64
Description               object
PurchasePrice            float64
ActualPrice              float64
Volume                    object
TotalPurchaseQuantity      int64
TotalPurchaseDollars     float64
TotalSalesQuantity       float64
TotalSalesDollars        float64
TotalExciseTax           float64
FreightCost              float64
dtype: object

In [12]:
vendor_sales_summary.isnull().sum()

VendorNumber                0
VendorName                  0
Brand                       0
Description                 0
PurchasePrice               0
ActualPrice                 0
Volume                      0
TotalPurchaseQuantity       0
TotalPurchaseDollars        0
TotalSalesQuantity       1633
TotalSalesDollars        1633
TotalExciseTax           1633
FreightCost                 0
dtype: int64

In [15]:
vendor_sales_summary['VendorName'].unique()

array(['BROWN-FORMAN CORP          ', 'DIAGEO NORTH AMERICA INC   ',
       'MARTIGNETTI COMPANIES', 'PERNOD RICARD USA          ',
       'BACARDI USA INC            ', 'JIM BEAM BRANDS COMPANY    ',
       'ULTRA BEVERAGE COMPANY LLP ', 'PROXIMO SPIRITS INC.       ',
       'MAJESTIC FINE WINES        ', 'STOLI GROUP,(USA) LLC      ',
       'CAMPARI AMERICA            ', 'SAZERAC CO INC             ',
       'MOET HENNESSY USA INC      ', 'M S WALKER INC             ',
       'CONSTELLATION BRANDS INC   ', 'SAZERAC NORTH AMERICA INC. ',
       'REMY COINTREAU USA INC     ', 'SIDNEY FRANK IMPORTING CO  ',
       'WILLIAM GRANT & SONS INC   ', 'PALM BAY INTERNATIONAL INC ',
       'E & J GALLO WINERY         ', 'HEAVEN HILL DISTILLERIES   ',
       'CASTLE BRANDS CORP.        ', 'SOUTHERN WINE & SPIRITS NE ',
       'DISARONNO INTERNATIONAL LLC', 'EDRINGTON AMERICAS         ',
       'TRINCHERO FAMILY ESTATES   ', 'STE MICHELLE WINE ESTATES  ',
       'WINE GROUP INC             ', 'P

In [16]:
vendor_sales_summary['Description'].unique()

array(['Jack Daniels No 7 Black', 'Capt Morgan Spiced Rum',
       "Tito's Handmade Vodka", ..., 'Smirnoff Light Strawberry',
       'Concannon Glen Ellen Wh Zin', 'Three Wishes Chard'], dtype=object)

In [17]:
vendor_sales_summary['Volume'] = vendor_sales_summary['Volume'].astype('float64')

In [19]:
vendor_sales_summary.fillna(0, inplace = True)

In [20]:
vendor_sales_summary['VendorName'] = vendor_sales_summary['VendorName'].str.strip()

In [21]:
vendor_sales_summary.dtypes

VendorNumber               int64
VendorName                object
Brand                      int64
Description               object
PurchasePrice            float64
ActualPrice              float64
Volume                   float64
TotalPurchaseQuantity      int64
TotalPurchaseDollars     float64
TotalSalesQuantity       float64
TotalSalesDollars        float64
TotalExciseTax           float64
FreightCost              float64
dtype: object

In [22]:
 vendor_sales_summary['GrossProfit'] = vendor_sales_summary['TotalSalesDollars'] - vendor_sales_summary['TotalPurchaseDollars']

In [23]:
 vendor_sales_summary['ProfitMargin'] =  (vendor_sales_summary['GrossProfit'] /vendor_sales_summary['TotalSalesDollars'])*100

In [26]:
 vendor_sales_summary['StockTurnover'] = vendor_sales_summary['TotalSalesQuantity'] /vendor_sales_summary['TotalPurchaseQuantity']

In [27]:
 vendor_sales_summary['StockPurchaseRatio'] = vendor_sales_summary['TotalSalesDollars']/vendor_sales_summary['TotalPurchaseDollars']

In [28]:
vendor_sales_summary.columns

Index(['VendorNumber', 'VendorName', 'Brand', 'Description', 'PurchasePrice',
       'ActualPrice', 'Volume', 'TotalPurchaseQuantity',
       'TotalPurchaseDollars', 'TotalSalesQuantity', 'TotalSalesDollars',
       'TotalExciseTax', 'FreightCost', 'GrossProfit', 'ProfitMargin',
       'StockTurnover', 'StockPurchaseRatio'],
      dtype='object')

In [10]:
cursor = conn.cursor()

In [15]:
cursor.execute("""
CREATE TABLE vendor_sales_summary (
   VendorNumber INT,
   VendorName VARCHAR(100),
   Brand INT,
   Description VARCHAR(100),
   PurchasePrice DECIMAL(10,2),
   ActualPrice DECIMAL(10,2),
   Volume INT,
   TotalPurchaseQuantity INT,
   TotalPurchaseDollars DECIMAL(15,2), 
   TotalSalesQuantity INT,
   TotalSalesDollars DECIMAL(15,2),
   TotalExciseTax DECIMAL(15,2),
   FreightCost DECIMAL(15,2),
   GrossProfit DECIMAL(15,2),
   ProfitMargin DECIMAL(15,2),
   StockTurnover DECIMAL(15,2),
   StockPurchaseRatio DECIMAL(15,2),
   PRIMARY KEY (VendorNumber, Brand)
);
""")

In [23]:
pd.read_sql_query( "select * from vendor_sales_summary", conn)

,VendorNumber,VendorName,Brand,Description,PurchasePrice,ActualPrice,Volume,TotalPurchaseQuantity,TotalPurchaseDollars,TotalSalesQuantity,TotalSalesDollars,TotalExciseTax,FreightCost
0,1128,BROWN-FORMAN CORP,1233,Jack Daniels No 7 Black,26.27,36.99,1750,60320,1584606.40,9578.0,344712.22,17598.14,68601.68
1,3960,DIAGEO NORTH AMERICA INC,4261,Capt Morgan Spiced Rum,16.17,22.99,1750,96073,1553500.41,20226.0,444810.74,37163.76,257032.07
2,4425,MARTIGNETTI COMPANIES,3405,Tito's Handmade Vodka,23.19,28.99,1750,62385,1446708.15,9203.0,275162.97,16909.12,144929.24
3,17035,PERNOD RICARD USA,8068,Absolut 80 Proof,18.24,24.99,1750,75385,1375022.40,11189.0,288135.11,20557.97,123780.22
4,3960,DIAGEO NORTH AMERICA INC,3545,Ketel One Vodka,21.89,29.99,1750,58783,1286759.87,11883.0,357759.17,21833.58,257032.07
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8507,9815,WINE GROUP INC,8527,Concannon Glen Ellen Wh Zin,1.32,4.99,750,2,2.64,3.0,5.97,0.33,27100.41
8508,8004,SAZERAC CO INC,5683,Dr McGillicuddy's Apple Pie,0.39,0.49,50,6,2.34,128.0,62.72,6.72,50293.62
8509,9815,WINE GROUP INC,22407,Three Wishes Chard,2.25,3.29,750,1,2.25,1.0,3.29,0.11,27100.41
8510,3960,DIAGEO NORTH AMERICA INC,3775,Smirnoff Sorbet Pine/Coconut,0.73,0.99,50,1,0.73,NaN,NaN,NaN,257032.07


In [22]:
vendor_sales_summary.to_sql('vendor_sales_summary',conn, if_exists ='replace', index = False)

In [24]:
import sqlite3
import pandas as pd
import logging
logging.basicConfig(
    filename="logs/get_vendor_summary.log", 
    level=logging.DEBUG,
    format="%(asctime)s - %(levelname)s - %(message)s", 
    filemode="a"  
)

def ingest_db(df, table_name, engine):
    '''this function will ingest the dataframe into database table'''
    df.to_sql(table_name, con = engine, if_exists = 'replace', index = False)
    
def create_vendor_summary(conn):
    '''this function will merge the different tables to get the overall vendor summary and adding new columns in the resultant data'''
    vendor_sales_summary = pd.read_sql_query("""WITH FreightSummary AS (
        SELECT 
            VendorNumber, 
            SUM(Freight) AS FreightCost 
        FROM vendor_invoice 
        GROUP BY VendorNumber
    ), 
    
    PurchaseSummary AS (
        SELECT 
            p.VendorNumber,
            p.VendorName,
            p.Brand,
            p.Description,
            p.PurchasePrice,
            pp.Price AS ActualPrice,
            pp.Volume,
            SUM(p.Quantity) AS TotalPurchaseQuantity,
            SUM(p.Dollars) AS TotalPurchaseDollars
        FROM purchases p
        JOIN purchase_prices pp
            ON p.Brand = pp.Brand
        WHERE p.PurchasePrice > 0
        GROUP BY p.VendorNumber, p.VendorName, p.Brand, p.Description, p.PurchasePrice, pp.Price, pp.Volume
    ), 
    
    SalesSummary AS (
        SELECT 
            VendorNo,
            Brand,
            SUM(SalesQuantity) AS TotalSalesQuantity,
            SUM(SalesDollars) AS TotalSalesDollars,
            SUM(SalesPrice) AS TotalSalesPrice,
            SUM(ExciseTax) AS TotalExciseTax
        FROM sales
        GROUP BY VendorNo, Brand
    ) 
    
    SELECT 
        ps.VendorNumber,
        ps.VendorName,
        ps.Brand,
        ps.Description,
        ps.PurchasePrice,
        ps.ActualPrice,
        ps.Volume,
        ps.TotalPurchaseQuantity,
        ps.TotalPurchaseDollars,
        ss.TotalSalesQuantity,
        ss.TotalSalesDollars,
        ss.TotalSalesPrice,
        ss.TotalExciseTax,
        fs.FreightCost
    FROM PurchaseSummary ps
    LEFT JOIN SalesSummary ss 
        ON ps.VendorNumber = ss.VendorNo 
        AND ps.Brand = ss.Brand
    LEFT JOIN FreightSummary fs 
        ON ps.VendorNumber = fs.VendorNumber
    ORDER BY ps.TotalPurchaseDollars DESC""",conn)

    return vendor_sales_summary


def clean_data(df):
    '''this function will clean the data'''
    # changing datatype to float
    df['Volume'] = df['Volume'].astype('float')
    
    # filling missing value with 0
    df.fillna(0,inplace = True)
    
    # removing spaces from categorical columns
    df['VendorName'] = df['VendorName'].str.strip()
    df['Description'] = df['Description'].str.strip()

    # creating new columns for better analysis
    df['GrossProfit'] = df['TotalSalesDollars'] - df['TotalPurchaseDollars']
    df['ProfitMargin'] = (df['GrossProfit'] / df['TotalSalesDollars'])*100
    df['StockTurnover'] = df['TotalSalesQuantity'] / df['TotalPurchaseQuantity']
    df['SalesToPurchaseRatio'] = df['TotalSalesDollars'] / df['TotalPurchaseDollars']

    
    return df

if __name__ == '__main__':
    # creating database connection
    conn = sqlite3.connect('inventory.db')
    
    logging.info('Creating Vendor Summary Table.....')
    summary_df = create_vendor_summary(conn)
    logging.info(summary_df.head())
    
    logging.info('Cleaning Data.....')
    clean_df = clean_data(summary_df)
    logging.info(clean_df.head())
    
    logging.info('Ingesting data.....')
    ingest_db(clean_df,'vendor_sales_summary',conn)
    logging.info('Completed')